# <p style="color:#07d839; font-family:sans-serif;"> 😀 Sentiment Analysis — Análisis de Sentimiento 🥲 </p>

---

## <p style="color:#07d839;">🎯 Objetivo de la Tarea</p>

El propósito de este análisis es desentrañar la carga emocional oculta tras las letras de diferentes géneros musicales. Buscamos responder preguntas clave sobre la naturaleza sentimental de la música:

*   **Polaridad:** ¿Qué géneros son predominantemente positivos, negativos o neutros?
*   **Intensidad:** ¿Existen géneros con una carga emocional negativa o positiva mucho más marcada?
*   **Emociones específicas:** ¿Qué géneros destacan en sentimientos como la **tristeza**, **ira**, **alegría** o **amor**?
*   **Variabilidad:** ¿Es el sentimiento constante dentro de un género o existe mucha disparidad entre canciones?

---

## <p style="color:#07d839;">⚙️ Metodología del Análisis</p>

La unidad básica de estudio es **cada canción individual**. El flujo de procesamiento sigue esta estructura lógica:

> **Flujo de Trabajo:**
> `Canción` ➔ `Letra (Texto)` ➔ `Modelo de Sentimiento` ➔ `Score (Positivo/Negativo/Neutro)` | `Top Sentimientos por género`



---

In [188]:
# librerías
import pandas as pd
import glob
import os
import numpy as np
import seaborn as sns
import re
import ast
import html
from pathlib import Path
import torch
import gc
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt


---
### 1. Limpieza dataset

+ Eliminar canciones sin lyrics.

+ Eliminar canciones sin ningún género válido.

+ Quitar letras demasiado cortas, por ejemplo menos de 30 o 50 palabras.

+ Normalizar nombres de géneros: pop, Pop, POP → pop.

+ Revisar duplicados por artist + song.

+ NLP sobre lyrics para preparar análisis de sentimiento 

Limpiar la letra sin destruir información útil para el sentimiento. En análisis de sentimiento no conviene eliminar stopwords agresivamente, porque palabras como not, never, no, without cambian completamente el significado.

In [189]:
# Fusionar Datasets - !!!!! SOLO EJECUTAR UNA VEZ !!!!!!

patron_busqueda = "music_dataset_*.csv"
archivos_csv = glob.glob(patron_busqueda)

# Verificamos
if not archivos_csv:
    print("No se encontraron archivos con el prefijo 'music_dataset_'. Revisa el directorio.")
else:
    print(f"Archivos encontrados para fusionar: {archivos_csv}")
    
    # Leer cada archivo CSV y almacenarlo en una lista
    lista_dataframes = []
    
    for archivo in archivos_csv:
        # Leemos el CSV
        df = pd.read_csv(archivo)
        lista_dataframes.append(df)
        print(f"Leído: {archivo} con {len(df)} filas.")

    # Concatenar (fusionar) todos los DataFrames de la lista en uno solo
    # ignore_index=True es importante para que el índice se reinicie (0, 1, 2...) 
    # en lugar de mantener los índices originales de cada archivo por separado.
    df_final = pd.concat(lista_dataframes, ignore_index=True)

    # Guardar el resultado en un nuevo archivo CSV
    nombre_archivo_salida = "music_dataset_final.csv"
    
    # index=False evita que se guarde la columna de números de fila en el CSV final
    df_final.to_csv(nombre_archivo_salida, index=False)

    print(f"\n¡Fusión completada! El archivo final tiene {len(df_final)} filas en total.")
    print(f"Se ha guardado como: {nombre_archivo_salida}")

Archivos encontrados para fusionar: ['music_dataset_20000_30000.csv', 'music_dataset_jose_2.csv', 'music_dataset_40000_50000.csv', 'music_dataset_30000_40000.csv', 'music_dataset_50000_70000.csv']
Leído: music_dataset_20000_30000.csv con 1035 filas.
Leído: music_dataset_jose_2.csv con 173 filas.
Leído: music_dataset_40000_50000.csv con 924 filas.
Leído: music_dataset_30000_40000.csv con 801 filas.
Leído: music_dataset_50000_70000.csv con 602 filas.

¡Fusión completada! El archivo final tiene 3535 filas en total.
Se ha guardado como: music_dataset_final.csv


In [108]:
# cargar dataset
DATA_PATH = "music_dataset_final.csv"

df = pd.read_csv(DATA_PATH)

print("Shape inicial:", df.shape)
display(df.head())
display(df.tail())

Shape inicial: (3535, 8)


,artist,song,lyrics,genres,genre_1,genre_2,genre_3,year
0,Tarja Turunen,Until Silence,NaN,"symphonic metal, female vocalists, female fron...",symphonic metal,female vocalists,female fronted metal,2004.0
1,Susan Boyle,You Have To Be There,What is it Lord that you want\n\nThat I am not...,"pop, love it, female vocalists",pop,love it,female vocalists,1995.0
2,Taylor Swift,Cruel Summer,"(Yeah, yeah, yeah, yeah)\n\n\n\nFever dream hi...","electropop, pop, synthpop",electropop,pop,synthpop,2023.0
3,Taylor Swift,London Boy,[Idris Elba and James Corden:]\n\nWe can go dr...,NaN,NaN,NaN,NaN,2019.0
4,Taylor Swift,Sugar,What a thing to see\n\nWhat a thing to be\n\nW...,NaN,NaN,NaN,NaN,2005.0


,artist,song,lyrics,genres,genre_1,genre_2,genre_3,year
3530,Kiss,See You Tonight,I know it's around\n\nI don't have any doubts ...,"hard rock, glam rock, classic rock",hard rock,glam rock,classic rock,NaN
3531,Kiss,Easy Thing,NaN,NaN,NaN,NaN,NaN,NaN
3532,Kiss,Is That You?,Cat's droolin' on the bar stool\n\nShake your ...,"hard rock, glam rock, classic rock",hard rock,glam rock,classic rock,NaN
3533,Kiss,Not For The Innocent,"I'm mean and I'm dirty, like none you've ever ...","hard rock, glam rock, classic rock",hard rock,glam rock,classic rock,NaN
3534,Kiss,Take It Off,Well my mind is gettin' dirty\n\nYeah around e...,"hard rock, glam rock, classic rock",hard rock,glam rock,classic rock,NaN


In [109]:
# Normalizar columnas
df.columns = df.columns.str.strip().str.lower()

expected_cols = ["artist", "song", "lyrics", "genres", "genre_1", "genre_2", "genre_3"]

missing_cols = [col for col in expected_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas en el dataset: {missing_cols}")

print("Columnas disponibles:")
print(df.columns.tolist())

Columnas disponibles:
['artist', 'song', 'lyrics', 'genres', 'genre_1', 'genre_2', 'genre_3', 'year']


#### **1.1 NLP**

**PREPROCESADO DE LYRICS**

In [110]:
def clean_lyrics(text):
    """
    Limpieza conservadora de letras para análisis emocional.
    No elimina stopwords ni aplica stemming/lemmatization.
    """
    
    if pd.isna(text):
        return np.nan
    
    text = str(text)
    
    # Decodificar entidades HTML
    text = html.unescape(text)
    
    # Eliminar URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    
    # Normalizar comillas raras
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
    )
    
    # Eliminar etiquetas típicas de letras
    section_patterns = [
        r"\[.*?(verse|chorus|bridge|intro|outro|hook|pre-chorus|post-chorus|interlude|refrain).*?\]",
        r"\(.*?(verse|chorus|bridge|intro|outro|hook|pre-chorus|post-chorus|interlude|refrain).*?\)"
    ]
    
    for pattern in section_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    
    # Eliminar patrones frecuentes de Genius u otras fuentes
    text = re.sub(r"\d+\s*contributors?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"you might also like", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"embed$", " ", text, flags=re.IGNORECASE)
    
    # Normalizar saltos de línea y tabs
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")
    
    # Espacios múltiples
    text = re.sub(r"\s+", " ", text).strip()
    
    if text == "":
        return np.nan
    
    return text

df["lyrics_clean"] = df["lyrics"].apply(clean_lyrics)

def count_words(text):
    if pd.isna(text):
        return 0
    return len(re.findall(r"\b[\w']+\b", str(text)))

df["word_count"] = df["lyrics_clean"].apply(count_words)

display(df[["artist", "song", "lyrics_clean", "word_count"]].head())

,artist,song,lyrics_clean,word_count
0,Tarja Turunen,Until Silence,NaN,0
1,Susan Boyle,You Have To Be There,What is it Lord that you want That I am not se...,325
2,Taylor Swift,Cruel Summer,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
3,Taylor Swift,London Boy,[Idris Elba and James Corden:] We can go drivi...,443
4,Taylor Swift,Sugar,What a thing to see What a thing to be What a ...,241


- No hemos eliminado stopwords.
- No hemos hecho stemming.
- No hemos hecho lemmatization.
- No hemos eliminado signos de puntuación de forma agresiva.
- No hemos convertido todo a minúsculas.
- No hemos eliminado negaciones como "not", "no", "never".
- No hemos tokenizado manualmente para entrenar un modelo clásico.

En la fase de preprocesado de letras se aplicó una limpieza conservadora orientada al uso posterior de un modelo Transformer de clasificación emocional. Se eliminaron elementos no pertenecientes a la letra, como URLs, etiquetas estructurales de canciones ([Verse], [Chorus], [Bridge]), patrones procedentes del scraping y espacios duplicados. También se normalizaron comillas y entidades HTML para mejorar la consistencia del texto.

No se aplicaron técnicas agresivas como eliminación de stopwords, stemming o lematización, ya que el modelo emocional necesita conservar el contexto lingüístico completo de las frases. Finalmente, se calculó el número de palabras de cada lyric y se descartaron canciones sin letra válida o con letras demasiado cortas.

Los modelos DeepLearning/Transformer utilizan el contexto de la oración completa para entender el significado, por lo que un preprocesado fuerte (como eliminar palabras comunes o signos de puntuación) destruye la estructura gramatical y los matices emocionales que necesitan para funcionar correctamente. En su lugar, requieren recibir el texto lo más natural posible, limitando la limpieza únicamente a la eliminación de ruido técnico como etiquetas o metadatos.

### **1.2.** **PREPROCESADO DE GENRES**

In [153]:
import pandas as pd
import numpy as np
import re
import ast
import html

# ============================================================
# 1. Géneros válidos
# ============================================================

VALID_GENRES = {
    "acid house",
    "acid jazz",
    "acid rock",
    "acid techno",
    "afro-cuban jazz",
    "afro-pop",
    "afrobeats",
    "alt rock",
    "alt-country",
    "alternative",
    "alternative hip hop",
    "alternative metal",
    "alternative r&b",
    "alternative rock",
    "alternative/modern metal",
    "amapiano",
    "ambient",
    "american country",
    "anti-folk",
    "appalachian",
    "arabic pop",
    "art pop",
    "art rock",
    "avant-garde",
    "avant-garde jazz",
    "bachata",
    "baroque",
    "bass music",
    "bassline",
    "bebop",
    "bhangra",
    "big band",
    "black metal",
    "bluegrass",
    "blues",
    "blues rock",
    "bollywood",
    "boogie",
    "boom bap",
    "bossa nova",
    "britpop",
    "bro-country",
    "brostep",
    "bubblegum pop",
    "c-pop",
    "calypso",
    "canadian country",
    "carnatic classical",
    "celtic",
    "chamber pop",
    "chicago blues",
    "chicago drill",
    "chillout",
    "chillstep",
    "chiptune (8-bit)",
    "choral & liturgical",
    "classic blues",
    "classic country",
    "classic metal",
    "classic r&b",
    "classic rock",
    "classical",
    "cloud rap",
    "conscious hip hop",
    "contemporary country",
    "contemporary folk",
    "contemporary pop",
    "contemporary r&b",
    "contry",
    "cool jazz",
    "country",
    "country and western",
    "country blues",
    "country pop",
    "country rap",
    "country rock",
    "country two step",
    "countrypolitan",
    "cumbia",
    "cyberpunk",
    "dance",
    "dance-pop",
    "dancehall",
    "dangdut",
    "death metal",
    "deathcore",
    "deep house",
    "delta blues",
    "detroit techno",
    "disco",
    "dixieland",
    "doo-wop",
    "doom metal",
    "downtempo",
    "dream pop",
    "drill",
    "drum and bass",
    "dub",
    "dubstep",
    "early jazz",
    "early music",
    "east coast hip hop",
    "electro house",
    "electro pop",
    "electro swing",
    "electroclash",
    "electronic",
    "electronic & dance (edm)",
    "electropop",
    "emo",
    "emo rap",
    "emocore",
    "enka",
    "experimental",
    "expressionism",
    "extreme metal",
    "folk",
    "folk pop",
    "folk rock",
    "folkrock",
    "freak folk",
    "free jazz",
    "french house",
    "frenchcore",
    "funk",
    "fusion",
    "future bass",
    "g-funk",
    "gabber",
    "gagaku",
    "gamelan",
    "gangsta rap",
    "garage rock",
    "glam metal",
    "glitch",
    "go-go",
    "goa trance",
    "gospel",
    "gothic metal",
    "grand opera",
    "gregorian chant",
    "grime",
    "grunge",
    "gypsy jazz",
    "hard bop",
    "hard dance",
    "hard rock",
    "hard trance",
    "hardcore",
    "hardcore punk",
    "hardstyle",
    "heartland rock",
    "heavy metal",
    "highlife",
    "hindustani classical",
    "hip hop",
    "honky-tonk",
    "house",
    "hyperpop",
    "idm (intelligent dance music)",
    "illbient",
    "impressionism",
    "indie",
    "indie folk",
    "indie pop",
    "indie rock",
    "indietronica",
    "industrial techno",
    "italo disco",
    "j-pop",
    "jazz",
    "jazz funk",
    "jazz rap",
    "jazz-rock",
    "jump blues",
    "jungle",
    "jùjú",
    "k-hip hop",
    "k-pop",
    "klezmer",
    "krautrock",
    "kwaito",
    "latin hip hop",
    "latin jazz",
    "latin pop",
    "latin trap",
    "liquid drum and bass",
    "lo-fi hip hop",
    "mariachi",
    "mass",
    "math rock",
    "mbalax",
    "medieval",
    "melodic death metal",
    "memphis blues",
    "merengue",
    "metal",
    "metalcore",
    "midwest hip hop",
    "minimal techno",
    "minimalism",
    "modern country",
    "motown",
    "mumble rap",
    "musique concrète",
    "neo-soul",
    "neoclassical",
    "neurofunk",
    "new age",
    "new jack swing",
    "new traditionalist",
    "noise",
    "noise rock",
    "norteño",
    "northern soul",
    "nu jazz",
    "nu-disco",
    "nu-metal",
    "nwobhm (new wave of british heavy metal)",
    "ny drill",
    "old-time",
    "opera",
    "opera buffa",
    "operetta",
    "outlaw country",
    "p-funk",
    "pansori",
    "phonk",
    "piedmont blues",
    "pinoy rock",
    "pop",
    "pop country",
    "pop punk",
    "pop rock",
    "pop soul",
    "post-hardcore",
    "post-minimalism",
    "post-punk",
    "post-rock",
    "power metal",
    "power pop",
    "progressive folk rock",
    "progressive house",
    "progressive rock",
    "progressive rock (prog)",
    "proto-punk",
    "psychedelic",
    "psychedelic rock",
    "psytrance",
    "punk rock",
    "qawwali",
    "r&b",
    "ragtime",
    "rai",
    "ranchera",
    "rap",
    "reggae",
    "reggaeton",
    "renaissance",
    "requiem",
    "retrowave",
    "riddim",
    "riot grrrl",
    "rock",
    "rock and roll",
    "rock n roll",
    "rockabilly",
    "rocksteady",
    "romantic",
    "roots rock",
    "salsa",
    "samba",
    "screamo",
    "serialism (12-tone)",
    "shoegaze",
    "ska",
    "ska punk",
    "skiffle",
    "slap house",
    "sludge",
    "smooth jazz",
    "soca",
    "soft rock",
    "sophisti-pop",
    "soukous",
    "soul",
    "soul blues",
    "southern hip hop",
    "southern rock",
    "space rock",
    "stax",
    "stoner metal",
    "sufi music",
    "surf rock",
    "swamp blues",
    "swing",
    "symphonic metal",
    "synth-pop",
    "synthpop",
    "synthwave",
    "tango",
    "tarab",
    "tech house",
    "techno",
    "teen pop",
    "tejano",
    "texas blues",
    "thrash metal",
    "traditional chinese instrumental",
    "traditional country",
    "traditional folk",
    "traditional pop",
    "trance",
    "trap",
    "trip-hop",
    "tropical house",
    "trot",
    "uk drill",
    "uk garage",
    "uk hip hop",
    "uplifting trance",
    "vaporwave",
    "west coast hip hop",
    "west coast jazz",
    "western swing",
    "zouk"
}

# ============================================================
# 2. Alias de normalización
# ============================================================

GENRE_ALIASES = {
    "hip-hop": "hip hop",
    "hiphop": "hip hop",

    "rnb": "r&b",
    "r and b": "r&b",
    "rhythm and blues": "r&b",

    "rock & roll": "rock",
    "rock n' roll": "rock and roll",

    "edm": "electronic",
    "electronic dance music": "electronic",

    "contry": "country",
    "country-pop": "country pop",
    "alt country": "alt-country",
    "honky tonk": "honky-tonk",

    "folkrock": "folk rock",
    "blues-rock": "blues rock",
    "jazz rock": "jazz-rock",

    "singer-songwriter": "singer songwriter",
    "neo soul": "neo-soul",
    "synthpop": "synth-pop",
    "trip hop": "trip-hop"
}

# ============================================================
# 3. Agrupación de subgéneros en macrogéneros
# ============================================================

GENRE_GROUPS = {
    "Rock": [
        "acid rock", "alt rock", "alternative", "alternative rock", "art rock",
        "blues rock", "britpop", "country rock", "folk rock", "garage rock",
        "grunge", "hard rock", "heartland rock", "indie", "indie rock",
        "math rock", "noise rock", "pinoy rock", "pop rock", "post-rock",
        "progressive rock", "progressive rock (prog)", "psychedelic",
        "psychedelic rock", "riot grrrl", "rock", "rock n roll",
        "rockabilly", "roots rock", "shoegaze", "soft rock",
        "southern rock", "space rock", "surf rock"
    ],

    "Pop": [
        "afro-pop", "arabic pop", "art pop", "bubblegum pop", "c-pop",
        "chamber pop", "contemporary pop", "country pop", "dance-pop",
        "dream pop", "electro pop", "electropop", "folk pop", "hyperpop",
        "indie pop", "j-pop", "k-pop", "latin pop", "pop", "power pop",
        "sophisti-pop", "synth-pop", "teen pop", "traditional pop"
    ],

    "Hip Hop / Rap": [
        "alternative hip hop", "boom bap", "chicago drill", "cloud rap",
        "conscious hip hop", "country rap", "drill", "east coast hip hop",
        "emo rap", "g-funk", "gangsta rap", "grime", "hip hop", "jazz rap",
        "k-hip hop", "latin hip hop", "latin trap", "lo-fi hip hop",
        "midwest hip hop", "mumble rap", "ny drill", "phonk", "rap",
        "southern hip hop", "trap", "uk drill", "uk hip hop",
        "west coast hip hop"
    ],

    "Electronic / Dance": [
        "acid house", "acid techno", "amapiano", "ambient", "bass music",
        "bassline", "brostep", "chillout", "chillstep", "chiptune (8-bit)",
        "cyberpunk", "dance", "deep house", "detroit techno", "disco",
        "downtempo", "drum and bass", "dubstep", "electro house",
        "electro swing", "electroclash", "electronic",
        "electronic & dance (edm)", "french house", "frenchcore",
        "future bass", "gabber", "glitch", "goa trance", "hard dance",
        "hard trance", "hardstyle", "house", "idm (intelligent dance music)",
        "illbient", "indietronica", "industrial techno", "italo disco",
        "jungle", "liquid drum and bass", "minimal techno", "neurofunk",
        "nu-disco", "progressive house", "psytrance", "retrowave", "riddim",
        "slap house", "synthwave", "tech house", "techno", "trance",
        "trip-hop", "tropical house", "uk garage", "uplifting trance",
        "vaporwave"
    ],

    "Jazz": [
        "acid jazz", "afro-cuban jazz", "avant-garde jazz", "bebop",
        "big band", "cool jazz", "dixieland", "early jazz", "free jazz",
        "gypsy jazz", "hard bop", "jazz", "jazz funk", "jazz-rock",
        "latin jazz", "nu jazz", "ragtime", "smooth jazz", "swing",
        "west coast jazz"
    ],

    "Blues": [
        "blues", "chicago blues", "classic blues", "country blues",
        "delta blues", "jump blues", "memphis blues", "piedmont blues",
        "soul blues", "swamp blues", "texas blues"
    ],

    "Country": [
        "alt-country", "american country", "appalachian", "bluegrass",
        "bro-country", "canadian country", "classic country",
        "contemporary country", "country", "country and western",
        "country two step", "countrypolitan", "honky-tonk",
        "modern country", "outlaw country", "pop country",
        "traditional country", "western swing"
    ],

    "R&B / Soul / Funk": [
        "alternative r&b", "boogie", "classic r&b", "contemporary r&b",
        "doo-wop", "funk", "go-go", "gospel", "motown", "neo-soul",
        "new jack swing", "northern soul", "p-funk", "pop soul",
        "r&b", "soul", "stax"
    ],

    "Metal": [
        "alternative metal", "alternative/modern metal", "black metal",
        "classic metal", "death metal", "deathcore", "doom metal",
        "extreme metal", "glam metal", "gothic metal", "heavy metal",
        "melodic death metal", "metal", "metalcore", "nu-metal",
        "nwobhm (new wave of british heavy metal)", "power metal",
        "sludge", "stoner metal", "symphonic metal", "thrash metal"
    ],

    "Folk / Traditional": [
        "anti-folk", "celtic", "contemporary folk", "folk", "freak folk",
        "indie folk", "old-time", "progressive folk rock", "skiffle",
        "traditional folk"
    ],

    "Classical / Art Music": [
        "avant-garde", "baroque", "carnatic classical", "choral & liturgical",
        "classical", "early music", "expressionism", "grand opera",
        "gregorian chant", "hindustani classical", "impressionism", "mass",
        "medieval", "minimalism", "musique concrète", "neoclassical",
        "opera", "opera buffa", "operetta", "post-minimalism",
        "renaissance", "requiem", "romantic", "serialism (12-tone)",
        "traditional chinese instrumental"
    ],

    "Latin": [
        "bachata", "bossa nova", "cumbia", "mariachi", "merengue",
        "norteño", "ranchera", "reggaeton", "salsa", "samba",
        "tango", "tejano"
    ],

    "Reggae / Caribbean": [
        "calypso", "dancehall", "dub", "reggae", "rocksteady",
        "ska", "soca", "zouk"
    ],

    "Punk / Emo / Hardcore": [
        "emo", "emocore", "hardcore", "hardcore punk", "pop punk",
        "post-hardcore", "post-punk", "proto-punk", "punk rock",
        "screamo", "ska punk"
    ],

    "Global / World": [
        "afrobeats", "bhangra", "bollywood", "dangdut", "enka",
        "gagaku", "gamelan", "highlife", "jùjú", "klezmer",
        "kwaito", "mbalax", "pansori", "qawwali", "rai",
        "soukous", "sufi music", "tarab", "trot"
    ],

    "Experimental / Other": [
        "experimental", "fusion", "krautrock", "new age",
        "new traditionalist", "noise"
    ]
}

# ============================================================
# 4. Crear diccionario inverso: subgénero -> macrogénero
# ============================================================

GENRE_TO_GROUP = {}

for group, genres in GENRE_GROUPS.items():
    for genre in genres:
        genre_norm = str(genre).strip().lower()
        genre_norm = GENRE_ALIASES.get(genre_norm, genre_norm)
        GENRE_TO_GROUP[genre_norm] = group

# ============================================================
# 5. Función para normalizar subgéneros
# ============================================================

def normalize_genre(genre):
    """
    Normaliza una etiqueta de género individual.
    Devuelve el subgénero normalizado si pertenece a VALID_GENRES.
    Si no pertenece, devuelve None.
    """

    if pd.isna(genre):
        return None

    genre = str(genre)
    genre = html.unescape(genre)
    genre = genre.strip().lower()

    # Quitar comillas externas
    genre = genre.strip("'").strip('"')

    # Normalizar espacios y separadores
    genre = genre.replace("_", " ")
    genre = re.sub(r"\s+", " ", genre).strip()

    # Quitar puntuación externa sin romper r&b, trip-hop o paréntesis internos
    genre = genre.strip(" .;:,")

    # Aplicar alias
    genre = GENRE_ALIASES.get(genre, genre)

    # Validar
    if genre not in VALID_GENRES:
        return None

    return genre

# ============================================================
# 6. Función para convertir subgénero a macrogénero
# ============================================================

def map_genre_to_group(genre):
    """
    Convierte un subgénero en macrogénero.
    """

    genre_norm = normalize_genre(genre)

    if genre_norm is None:
        return None

    return GENRE_TO_GROUP.get(genre_norm, "Other")

# ============================================================
# 7. Parsear celda y devolver macrogéneros
# ============================================================

def parse_genre_cell(value):
    """
    Convierte una celda de género en una lista de macrogéneros.

    Soporta:
    - valor simple: "pop"
    - lista en string: "['pop', 'rock']"
    - valores separados por comas: "pop, rock, r&b"

    Devuelve:
    - ["Pop"]
    - ["Rock"]
    - ["Hip Hop / Rap", "Pop"]
    """

    if pd.isna(value):
        return []

    value = str(value).strip()

    parsed_values = []

    # Caso lista en string: "['pop', 'rock']"
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, (list, tuple, set)):
                parsed_values = list(parsed)
            else:
                parsed_values = [value]

        except Exception:
            parsed_values = [value]

    # Caso separado por comas: "pop, rock, r&b"
    elif "," in value:
        parsed_values = value.split(",")

    # Caso simple
    else:
        parsed_values = [value]

    genre_groups = []

    for genre in parsed_values:
        group = map_genre_to_group(genre)

        if group is not None:
            genre_groups.append(group)

    # Eliminar duplicados manteniendo el orden
    unique_groups = []
    seen = set()

    for group in genre_groups:
        if group not in seen:
            unique_groups.append(group)
            seen.add(group)

    return unique_groups

# ============================================================
# 8. Aplicar a genre_1, genre_2 y genre_3
# ============================================================

genre_cols = ["genre_1", "genre_2", "genre_3"]

for col in genre_cols:
    df[col + "_clean"] = df[col].apply(
        lambda x: parse_genre_cell(x)[0] if len(parse_genre_cell(x)) > 0 else None
    )

# ============================================================
# 9. Construir lista final de macrogéneros por canción
# ============================================================

def build_genre_list(row):
    """
    Construye una lista de macrogéneros únicos por canción.
    """

    genres = []

    for col in ["genre_1_clean", "genre_2_clean", "genre_3_clean"]:
        genre = row[col]

        if genre is not None and pd.notna(genre):
            genres.append(genre)

    unique_genres = []
    seen = set()

    for genre in genres:
        if genre not in seen:
            unique_genres.append(genre)
            seen.add(genre)

    return unique_genres


df["genre_list"] = df.apply(build_genre_list, axis=1)
df["n_genres"] = df["genre_list"].apply(len)

# ============================================================
# 10. Comprobaciones rápidas
# ============================================================

print("Ejemplo parse_genre_cell('alternative rock'):", parse_genre_cell("alternative rock"))
print("Ejemplo parse_genre_cell('pop, r&b, trap'):", parse_genre_cell("pop, r&b, trap"))

print("\nNúmero de canciones sin macrogénero válido:")
print((df["n_genres"] == 0).sum())

print("\nDistribución de macrogéneros:")
display(
    df.explode("genre_list")["genre_list"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "genre_group", "genre_list": "n_songs"})
)

display(
    df[[
        "artist",
        "song",
        "genre_1",
        "genre_2",
        "genre_3",
        "genre_1_clean",
        "genre_2_clean",
        "genre_3_clean",
        "genre_list",
        "n_genres"
    ]].head(20)
)

Ejemplo parse_genre_cell('alternative rock'): ['Rock']
Ejemplo parse_genre_cell('pop, r&b, trap'): ['Pop', 'R&B / Soul / Funk', 'Hip Hop / Rap']

Número de canciones sin macrogénero válido:
592

Distribución de macrogéneros:


,n_songs,count
0,Rock,792
1,Pop,789
2,Country,778
3,Electronic / Dance,771
4,Folk / Traditional,355
5,Other,268
6,R&B / Soul / Funk,221
7,Punk / Emo / Hardcore,153
8,Metal,140
9,Hip Hop / Rap,90


,artist,song,genre_1,genre_2,genre_3,genre_1_clean,genre_2_clean,genre_3_clean,genre_list,n_genres
0,Tarja Turunen,Until Silence,symphonic metal,female vocalists,female fronted metal,Metal,NaN,NaN,[Metal],1
1,Susan Boyle,You Have To Be There,pop,love it,female vocalists,Pop,NaN,NaN,[Pop],1
2,Taylor Swift,Cruel Summer,electropop,pop,synthpop,Pop,Pop,Pop,[Pop],1
3,Taylor Swift,London Boy,NaN,NaN,NaN,NaN,NaN,NaN,[],0
4,Taylor Swift,Sugar,NaN,NaN,NaN,NaN,NaN,NaN,[],0
5,Shania Twain,Forever And For Always,country,pop,female vocalists,Country,Pop,NaN,"[Country, Pop]",2
6,Shania Twain,Wild and Wicked,NaN,NaN,NaN,NaN,NaN,NaN,[],0
7,Johnny Cash,A Croft In Clachan,NaN,NaN,NaN,NaN,NaN,NaN,[],0
8,Johnny Cash,Devil's Right Hand,country,folk,singer-songwriter,Country,Folk / Traditional,NaN,"[Country, Folk / Traditional]",2
9,Johnny Cash,I Heard The Bells On Christmas Day,christmas,country,folk,NaN,Country,Folk / Traditional,"[Country, Folk / Traditional]",2


In [154]:
# NORMALIZAR GENERO

genre_cols = ["genre_1", "genre_2", "genre_3"]

for col in genre_cols:
    df[col + "_clean"] = df[col].apply(
        lambda x: parse_genre_cell(x)[0] if len(parse_genre_cell(x)) > 0 else None
    )

display(df[["genres", "genre_1", "genre_2", "genre_3", 
            "genre_1_clean", "genre_2_clean", "genre_3_clean"]].head())

,genres,genre_1,genre_2,genre_3,genre_1_clean,genre_2_clean,genre_3_clean
0,"symphonic metal, female vocalists, female fron...",symphonic metal,female vocalists,female fronted metal,Metal,NaN,NaN
1,"pop, love it, female vocalists",pop,love it,female vocalists,Pop,NaN,NaN
2,"electropop, pop, synthpop",electropop,pop,synthpop,Pop,Pop,Pop
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [155]:
MIN_WORDS = 1

initial_rows = len(df)

mask_valid_lyrics = df["lyrics_clean"].notna() & (df["word_count"] >= MIN_WORDS)
mask_valid_genres = df["n_genres"] > 0

df_clean = df[mask_valid_lyrics & mask_valid_genres].copy()

print("Filas iniciales:", initial_rows)
print("Filas tras limpieza:", len(df_clean))
print("Canciones eliminadas:", initial_rows - len(df_clean))

print("\nMotivos principales:")
print("Sin letra válida:", (~mask_valid_lyrics).sum())
print("Sin género válido:", (~mask_valid_genres).sum())

Filas iniciales: 3535
Filas tras limpieza: 2225
Canciones eliminadas: 1310

Motivos principales:
Sin letra válida: 1000
Sin género válido: 592


In [156]:
def normalize_text_id(text):
    """
    Normaliza texto para detectar duplicados.
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text).strip().lower()
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text)
    
    return text


df_clean["artist_norm"] = df_clean["artist"].apply(normalize_text_id)
df_clean["song_norm"] = df_clean["song"].apply(normalize_text_id)

duplicates = df_clean.duplicated(subset=["artist_norm", "song_norm"], keep=False)

print("duplicados:", duplicates.sum())

df_clean = (
    df_clean
    .sort_values("word_count", ascending=False)
    .drop_duplicates(subset=["artist_norm", "song_norm"], keep="first")
    .sort_index()
    .copy()
)

print("Shape tras eliminar duplicados:", df_clean.shape)

duplicados: 104
Shape tras eliminar duplicados: (2173, 17)


In [157]:
# Dataset expandido por genero

df_genres = df_clean.explode("genre_list").copy()

df_genres = df_genres.rename(columns={"genre_list": "genre"})

df_genres = df_genres[df_genres["genre"].notna()].copy()

# Evitar que una misma canción aparezca dos veces en el mismo género
df_genres = df_genres.drop_duplicates(
    subset=["artist_norm", "song_norm", "genre"],
    keep="first"
)

print("Shape df_clean, una fila por canción:", df_clean.shape)
print("Shape df_genres, una fila por canción-género:", df_genres.shape)

display(df_genres[["artist", "song", "genre", "lyrics_clean", "word_count"]].head())

Shape df_clean, una fila por canción: (2173, 17)
Shape df_genres, una fila por canción-género: (3288, 17)


,artist,song,genre,lyrics_clean,word_count
1,Susan Boyle,You Have To Be There,Pop,What is it Lord that you want That I am not se...,325
2,Taylor Swift,Cruel Summer,Pop,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
5,Shania Twain,Forever And For Always,Country,"Mm, mm Mm, in your arms Oh I can hear your hea...",389
5,Shania Twain,Forever And For Always,Pop,"Mm, mm Mm, in your arms Oh I can hear your hea...",389
9,Johnny Cash,I Heard The Bells On Christmas Day,Country,I heard the bells on Christmas Day Their old f...,169


In [158]:
genre_counts = df_genres["genre"].value_counts()


# 1. Le decimos a Pandas que no ponga límite máximo de filas
#pd.set_option('display.max_rows', None)

# 2. Mostramos tu tabla (sin el .head())
display(
    genre_counts
    .reset_index()
    .rename(columns={"index": "genre", "genre": "n_songs"})
)

# 3. Restauramos el límite normal (por defecto suele ser 60)
# Esto es importante para que futuras tablas grandes no te bloqueen la pantalla
pd.reset_option('display.max_rows')

print("Número total de géneros únicos:", df_genres["genre"].nunique())
print("Número total de registros canción-género:", len(df_genres))


,n_songs,count
0,Rock,655
1,Electronic / Dance,565
2,Pop,542
3,Country,524
4,Folk / Traditional,257
5,Other,207
6,R&B / Soul / Funk,164
7,Punk / Emo / Hardcore,144
8,Metal,114
9,Hip Hop / Rap,51


Número total de géneros únicos: 16
Número total de registros canción-género: 3288


In [159]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

#df_clean.to_csv(OUTPUT_DIR / "songs_clean_by_song.csv", index=False)
df_genres.to_csv(OUTPUT_DIR / "songs_clean_by_genre_exploded.csv", index=False)

print("Archivos guardados:")
print(OUTPUT_DIR / "songs_clean_by_song.csv")
print(OUTPUT_DIR / "songs_clean_by_genre_exploded.csv")


Archivos guardados:
outputs/songs_clean_by_song.csv
outputs/songs_clean_by_genre_exploded.csv


En primer lugar, se realizó una limpieza del dataset original con el objetivo de preparar las letras y los géneros musicales para el análisis emocional posterior. Se eliminaron canciones sin letra disponible, canciones con letras demasiado cortas y registros sin ningún género válido. La limpieza de las letras fue conservadora, evitando eliminar stopwords o modificar excesivamente el texto, ya que el modelo Deep Learning utilizado posteriormente necesita conservar el contexto lingüístico de las frases.

Además, se normalizaron las etiquetas de género presentes en las columnas genre_1, genre_2 y genre_3, convirtiéndolas a minúsculas, eliminando espacios innecesarios y unificando algunas variantes frecuentes. Dado que una canción puede pertenecer a varios géneros, se generó un segundo dataset en formato expandido, donde cada fila representa una combinación canción-género. Este formato permite analizar posteriormente la distribución emocional de las letras para cada género musical.

| Archivo                                      | Nivel                       | Uso principal                   |
| -------------------------------------------- | --------------------------- | ------------------------------- |
| `songs_clean_by_song.csv`                    | Una fila por canción        | Aplicar el modelo emocional     |
| `songs_clean_by_genre_exploded.csv`          | Una fila por canción-género | Análisis completo por género    |



---
### **2. Clasificación de canciones en sentimiento.**

`Modelo 1`: modelo de sentimiento con 3 clases. Un modelo como cardiffnlp/twitter-roberta-base-sentiment-latest realiza la tarea más fundamental (análisis de polaridad), clasificando el texto en tres categorías básicas: positivo, negativo o neutral.

`Modelo 2`: modelo de emociones con 7 clases. Un modelo como j-hartmann/emotion-english-distilroberta-base sube un nivel en complejidad al clasificar el texto en emociones básicas de Ekman (etiqueta única o single-label): anger (ira), disgust (asco), fear (miedo), joy (alegría), neutral, sadness (tristeza) y surprise (sorpresa).

`Modelo 3`: modelo GoEmotions con 28 emociones. Un modelo como SamLowe/roberta-base-go_emotions representa la mayor complejidad y granularidad. Está entrenado sobre el dataset GoEmotions y funciona como clasificación multi-label; es decir, entiende que los sentimientos humanos son complejos y permite que una misma letra active varias emociones simultáneas (como amor, optimismo y gratitud al mismo tiempo).

Proceso a seguir:

1. Cargar dataset
2. Aplicar modelos seleccionados
3. Analizar emociones por género

In [118]:
#!pip install transformers torch tqdm -q

In [143]:
# cargar datoset limpio por canción

df_songs = df_clean.copy()

print("Shape:", df_songs.shape)
#display(df_songs[["artist", "song", "lyrics_clean", "word_count", "genre_list"]].head())

df_songs = df_songs[df_songs["lyrics_clean"].notna()].copy()
df_songs = df_songs.reset_index(drop=True)

print("Canciones válidas:", len(df_songs))

Shape: (2173, 17)
Canciones válidas: 2173


In [ ]:
# CONFIGURACIÓN GENERAL MODELOS que vamos a usar para análisis

TEXT_COL = "lyrics_clean"

MODEL_CONFIGS = [
    {
        "model_key": "cardiff",
        "model_name": "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "task_type": "single_label",
        "output_col": "cardiff_sentiment",
        "score_col": "cardiff_sentiment_score",
        "top_k": 1,
        "exclude_neutral": True
    },
    {
        "model_key": "jhartmann",
        "model_name": "j-hartmann/emotion-english-distilroberta-base",
        "task_type": "single_label",
        "output_col": "jhartmann_emotion",
        "score_col": "jhartmann_emotion_score",
        "top_k": 1,
        "exclude_neutral": True
    },
    {
        "model_key": "goemotions",
        "model_name": "SamLowe/roberta-base-go_emotions",
        "task_type": "multi_label",
        "output_col": "goemotions_top3",
        "score_col": "goemotions_top3_scores",
        "top_k": 3,
        "exclude_neutral": True,
        "exclude_approval": True
    }
]

Excluimos la etiqueta "neutral" en todos los modelos porque la música es intrínsecamente expresiva; ignorarla evita que el peso emocional de la canción se diluya con frases puramente descriptivas. Por su parte, descartamos "approval" en GoEmotions porque palabras como "yeah" o "ok", que en las canciones actúan como simples muletillas rítmicas, generan falsos positivos al ser interpretadas erróneamente como actos de aprobación. Con este filtro contextual, obligamos a los modelos a capturar exclusivamente el verdadero núcleo sentimental de la obra.

In [121]:
# DISPOSITIVO DE PROCESAMIENTO 
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


device = get_device()


In [122]:
# FUNCIONES GENERALES - DIVIDIR LETRAS LARGAS EN CHUNKS
MAX_TOKENS = 500
STRIDE = 250


def split_text_into_chunks(text, tokenizer, max_tokens=MAX_TOKENS, stride=STRIDE):
    """
    Divide una letra larga en fragmentos de tokens.
    Esto evita que el modelo analice solo el principio de la canción.
    """

    if pd.isna(text) or str(text).strip() == "":
        return []

    text = str(text)

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) == 0:
        return []

    chunks = []
    step = max_tokens - stride

    for start in range(0, len(token_ids), step):
        end = start + max_tokens
        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        if chunk_text.strip():
            chunks.append(chunk_text.strip())

        if end >= len(token_ids):
            break

    return chunks



# NORMALIZAR ETIQUETAS DE LOS MODELOS

def normalize_label(label, model_key=None):
    """
    Normaliza etiquetas devueltas por los modelos.
    """

    label = str(label).lower().strip()

    # Por seguridad, si algún modelo usa LABEL_0 / LABEL_1
    if model_key == "cardiff":
        label_mapping = {
            "label_0": "negative",
            "label_1": "positive",
            "0": "negative",
            "1": "positive"
        }
        label = label_mapping.get(label, label)

    return label



# OBTENER LOS SCORES DE UNA CANCIÓN


def predict_text_scores(
    text,
    tokenizer,
    model,
    labels,
    device,
    task_type,
    batch_size=8
):
    """
    Aplica un modelo a una canción completa.
    1. Divide la letra en chunks.
    2. Predice scores por chunk.
    3. Promedia los scores para obtener una predicción global por canción.
    """

    chunks = split_text_into_chunks(text, tokenizer)

    if len(chunks) == 0:
        return {
            "scores": {label: np.nan for label in labels},
            "n_chunks": 0
        }

    all_scores = []

    for i in range(0, len(chunks), batch_size):
        batch_chunks = chunks[i:i + batch_size]

        inputs = tokenizer(
            batch_chunks,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

            if task_type == "single_label":
                probs = torch.softmax(outputs.logits, dim=1)

            elif task_type == "multi_label":
                probs = torch.sigmoid(outputs.logits)

            else:
                raise ValueError(f"task_type no reconocido: {task_type}")

        all_scores.append(probs.detach().cpu().numpy())

    all_scores = np.vstack(all_scores)
    mean_scores = all_scores.mean(axis=0)

    score_dict = {
        label: float(score)
        for label, score in zip(labels, mean_scores)
    }

    return {
        "scores": score_dict,
        "n_chunks": len(chunks)
    }




# EXTRAER LA PREDICCIÓN FINAL


def extract_prediction_from_scores(
    scores,
    model_key,
    top_k=1,
    exclude_neutral=False,
    exclude_approval=False
):
    """
    Extrae la predicción final a partir de los scores medios.
    
    Para car y j-hartmann:
    - devuelve top 1.
    
    Para GoEmotions:
    - devuelve top K emociones.
    """

    valid_scores = {
        emotion: score
        for emotion, score in scores.items()
        if not pd.isna(score)
    }

    if exclude_neutral:
        valid_scores = {
            emotion: score
            for emotion, score in valid_scores.items()
            if emotion != "neutral"
        }

    if exclude_approval:
        valid_scores = {
            emotion: score
            for emotion, score in valid_scores.items()
            if emotion != "approval"
        }

    if len(valid_scores) == 0:
        if top_k == 1:
            return None, np.nan
        else:
            return "", ""

    sorted_scores = sorted(
        valid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    if top_k == 1:
        top_label, top_score = sorted_scores[0]
        return top_label, top_score

    top_items = sorted_scores[:top_k]

    top_labels = [label for label, score in top_items]
    top_scores = [round(score, 4) for label, score in top_items]

    return ", ".join(top_labels), ", ".join(map(str, top_scores))





**`split_text_into_chunks`**: Divide la canción en fragmentos superpuestos de tokens para superar el límite de 512 de los Transformers. Usa una ventana deslizante ("stride") para que los bloques se solapen, asegurando que no se pierda el contexto gramatical ni el sentido emocional en los puntos de corte de la letra.

**`normalize_label`**: Limpia y estandariza las etiquetas devueltas por los modelos (minúsculas y sin espacios). Incluye una regla específica para el modelo CardiffNLP: traduce formatos crudos o numéricos ambiguos como "label_0" o "1" a sus equivalentes "negative" o "positive", unificando el formato de salida.

**`predict_text_scores`**: Es el motor de inferencia. Envía los fragmentos de la canción al modelo por lotes. Aplica funciones matemáticas (Softmax o Sigmoide según el tipo de tarea) para calcular probabilidades de 0 a 1. Finalmente, promedia las puntuaciones de todos los fragmentos para dar una evaluación global.

**`extract_prediction_from_scores`**: Filtra las probabilidades promediadas eliminando emociones no deseadas ("neutral" o "approval"). Luego, ordena los resultados por puntuación y devuelve la emoción ganadora (Top-1) o una lista de las principales separadas por comas (Top-K, para GoEmotions), listas para guardar en tu dataset.

In [ ]:
# FUNCIÓN PARA EJECUTAR CUALQUIER MODELO
# POSTERIORMENTE HAREMOS QUE SE APLIQUE A LOS 3 MODELOS QUE QUEREMOS COMPARAR


def run_model_on_df(
    df,
    model_config,
    text_col=TEXT_COL,
    device=device,
    batch_size=8,
    save_all_scores=False
):
    """
    Aplica un modelo Transformer a todas las canciones de df.
    
    Devuelve un DataFrame con:
    - predicción principal
    - score principal
    - número de chunks
    - opcionalmente, scores de todas las etiquetas
    """

    model_key = model_config["model_key"]
    model_name = model_config["model_name"]
    task_type = model_config["task_type"]
    output_col = model_config["output_col"]
    score_col = model_config["score_col"]
    top_k = model_config["top_k"]
    exclude_neutral = model_config["exclude_neutral"]

    print("=" * 90)
    print(f"Cargando modelo: {model_name}")
    print("=" * 90)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    model.to(device)
    model.eval()

    id2label = model.config.id2label

    labels = [
        normalize_label(id2label[i], model_key=model_key)
        for i in range(len(id2label))
    ]

    print("Etiquetas del modelo:")
    print(labels)

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Aplicando {model_key}"):

        text = row[text_col]

        prediction_result = predict_text_scores(
            text=text,
            tokenizer=tokenizer,
            model=model,
            labels=labels,
            device=device,
            task_type=task_type,
            batch_size=batch_size
        )

        scores = prediction_result["scores"]
        n_chunks = prediction_result["n_chunks"]

        pred_label, pred_score = extract_prediction_from_scores(
            scores=scores,
            model_key=model_key,
            top_k=top_k,
            exclude_neutral=exclude_neutral
        )

        result = {
            output_col: pred_label,
            score_col: pred_score,
            f"{model_key}_n_chunks": n_chunks
        }

        if save_all_scores:
            for label, score in scores.items():
                result[f"{model_key}_{label}"] = score

        results.append(result)

    df_results = pd.DataFrame(results)

    # Liberar memoria
    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return df_results

**`run_model_on_df`**: Orquesta el flujo completo. Carga el modelo en la GPU y procesa el dataset iterando cada canción con las funciones previas para predecir su emoción. Tras guardar los resultados en un DataFrame, libera exhaustivamente la memoria de la tarjeta gráfica para evitar cuelgues.

In [124]:
# APLICAR A LOS 3 MODELOS

df_model_comparison = df_songs.copy().reset_index(drop=True)

# Aseguramos que solo analizamos canciones con letra limpia válida
df_model_comparison = df_model_comparison[
    df_model_comparison[TEXT_COL].notna()
].copy()

df_model_comparison = df_model_comparison.reset_index(drop=True)

print("Canciones a analizar:", len(df_model_comparison))

Canciones a analizar: 2173


In [125]:
# EJECUTAR

for config in MODEL_CONFIGS:

    model_results = run_model_on_df(
        df=df_model_comparison,
        model_config=config,
        text_col=TEXT_COL,
        device=device,
        batch_size=8,
        save_all_scores=False
    )

    df_model_comparison = pd.concat(
        [
            df_model_comparison.reset_index(drop=True),
            model_results.reset_index(drop=True)
        ],
        axis=1
    )

    display(
        df_model_comparison[
            [
                "artist",
                "song",
                config["output_col"],
                config["score_col"],
                f"{config['model_key']}_n_chunks"
            ]
        ].head()
    )

Cargando modelo: cardiffnlp/twitter-roberta-base-sentiment-latest


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 47061.24it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Etiquetas del modelo:
['negative', 'neutral', 'positive']


Aplicando cardiff: 100%|██████████| 2173/2173 [01:30<00:00, 23.89it/s]


,artist,song,cardiff_sentiment,cardiff_sentiment_score,cardiff_n_chunks
0,Susan Boyle,You Have To Be There,negative,0.611403,1
1,Taylor Swift,Cruel Summer,negative,0.349225,2
2,Shania Twain,Forever And For Always,positive,0.850984,1
3,Johnny Cash,I Heard The Bells On Christmas Day,positive,0.297138,1
4,Johnny Cash,Southwind,negative,0.371507,1


...

Cargando modelo: SamLowe/roberta-base-go_emotions


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7031.91it/s]


Etiquetas del modelo:
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


Aplicando goemotions: 100%|██████████| 2173/2173 [01:23<00:00, 26.12it/s]


,artist,song,goemotions_top3,goemotions_top3_scores,goemotions_n_chunks
0,Susan Boyle,You Have To Be There,"confusion, curiosity, sadness","0.4627, 0.221, 0.2147",1
1,Taylor Swift,Cruel Summer,"sadness, annoyance, disappointment","0.262, 0.1959, 0.131",2
2,Shania Twain,Forever And For Always,"love, desire, caring","0.7028, 0.2402, 0.2331",1
3,Johnny Cash,I Heard The Bells On Christmas Day,"disappointment, sadness, realization","0.4433, 0.2922, 0.0573",1
4,Johnny Cash,Southwind,"excitement, joy, amusement","0.0542, 0.0256, 0.0177",1


In [126]:
display(
    df_model_comparison[
        [
            "artist",
            "song",
            "genre_list",
            "cardiff_sentiment",
            
            "jhartmann_emotion",
            
            "goemotions_top3",
            
        ]
    ].tail(20)
)

,artist,song,genre_list,cardiff_sentiment,jhartmann_emotion,goemotions_top3
2153,Aerosmith,Taste Of India,"[Rock, Other]",positive,fear,"love, admiration, approval"
2154,Iron Maiden,Hell On Earth,[Metal],negative,sadness,"desire, optimism, disappointment"
2155,Iron Maiden,Face In The Sand,[Metal],negative,fear,"curiosity, confusion, disappointment"
2156,Iron Maiden,Only The Good Die Young,[Metal],negative,anger,"curiosity, confusion, remorse"
2157,Iron Maiden,The Great Unknown,[Metal],negative,fear,"fear, realization, sadness"
2158,Led Zeppelin,You Shook Me,"[Other, Rock]",positive,fear,"sadness, love, annoyance"
2159,Led Zeppelin,Sick Again,"[Rock, Other]",positive,surprise,"excitement, approval, curiosity"
2160,Slipknot,All Hope Is Gone,[Metal],negative,anger,"annoyance, curiosity, anger"
2161,Slipknot,No Life,[Metal],negative,anger,"annoyance, disapproval, anger"
2162,Black Sabbath,Black Sabbath,"[Metal, Rock]",negative,fear,"fear, confusion, nervousness"


---

### **3. Agregación por género.**

Después de calcular el sentimiento por canción, se agrupa por género.


#### **3.1. Unir resultados de modelos con dataset expandido por género**

In [144]:
# PRIMERO UNIMOS LOS RESULTADOS DE LOS 3 MODELOS EN UN SOLO DATAFRAME EXPANDIDO POR GÉNERO
#df_model_comparison + outputs/songs_clean_by_genre_exploded_filtered.csv

df_genres_filtered = pd.read_csv("outputs/songs_clean_by_genre_exploded.csv")

print("df_model_comparison:", df_model_comparison.shape)
print("df_genres_filtered:", df_genres_filtered.shape)




# unimos las columnas que nos interesan del model_comparison con el df_genres_filtered 
# para tener un df completo con género + resultados de los modelos

model_cols = [
    "artist_norm",
    "song_norm",
    "cardiff_sentiment",
    "cardiff_sentiment_score",
    "jhartmann_emotion",
    "jhartmann_emotion_score",
    "goemotions_top3",
    "goemotions_top3_scores"
]

model_cols = [col for col in model_cols if col in df_model_comparison.columns]

df_predictions = df_model_comparison[model_cols].copy()

print("Columnas seleccionadas:", df_predictions.columns.tolist())





# HACEMOS MERGE DE LOS MODELOS CON EL DF DE GÉNEROS 
# PARA TENER UN DF COMPLETO CON ARTISTA, CANCIÓN, GÉNERO Y RESULTADOS DE LOS MODELOS

df_song_genre_models = df_genres_filtered.merge(
    df_predictions,
    on=["artist_norm", "song_norm"],
    how="left"
)

print("Shape tras merge:", df_song_genre_models.shape)



df_model_comparison: (2173, 26)
df_genres_filtered: (3288, 17)
Columnas seleccionadas: ['artist_norm', 'song_norm', 'cardiff_sentiment', 'cardiff_sentiment_score', 'jhartmann_emotion', 'jhartmann_emotion_score', 'goemotions_top3', 'goemotions_top3_scores']
Shape tras merge: (3288, 23)


Este bloque integra las predicciones de los tres modelos con la información de los géneros musicales. Primero, selecciona estrictamente las columnas de resultados (Cardiff, j-hartmann y GoEmotions). Luego, usa el artista y la canción como claves para realizar un cruce (`merge`) con el dataset de géneros, creando una base de datos maestra lista para analizar qué emociones predominan en cada estilo musical.

#### **3.2. Crear score numérico de sentimiento**

Crear un score numérico continuo basado en la polaridad del sentimiento nos permite interpretar y graficar fácilmente los resultados (creando un eje que va de -1 a +1).

Usaremos la siguiente lógica matemática:

| Etiqueta | Lógica | Ejemplo de Salida | Score Final |
| :--- | :--- | :--- | :--- |
| **Positive** | Score positivo | Positive (0.91) | `+0.91` |
| **Negative** | Score negativo | Negative (0.88) | `-0.88` |


In [128]:

def sentiment_to_signed_score(row, sentiment_col, score_col):
    label = row[sentiment_col]
    score = row[score_col]

    if pd.isna(label) or pd.isna(score):
        return np.nan

    label = str(label).lower()

    if label == "positive":
        return float(score)

    elif label == "negative":
        return -float(score)

    elif label == "neutral":
        return 0.0

    else:
        return np.nan


df_song_genre_models["sentiment_signed_score"] = df_song_genre_models.apply(
    lambda row: sentiment_to_signed_score(row, "cardiff_sentiment", "cardiff_sentiment_score"),
    axis=1
)

display(
    df_song_genre_models[
        [
            "artist",
            "song",
            "genre",
            "cardiff_sentiment",
            "cardiff_sentiment_score",
            "sentiment_signed_score"
        ]
    ].head(20)
)


,artist,song,genre,cardiff_sentiment,cardiff_sentiment_score,sentiment_signed_score
0,Susan Boyle,You Have To Be There,Pop,negative,0.611403,-0.611403
1,Taylor Swift,Cruel Summer,Pop,negative,0.349225,-0.349225
2,Shania Twain,Forever And For Always,Country,positive,0.850984,0.850984
3,Shania Twain,Forever And For Always,Pop,positive,0.850984,0.850984
4,Johnny Cash,I Heard The Bells On Christmas Day,Country,positive,0.297138,0.297138
5,Johnny Cash,I Heard The Bells On Christmas Day,Folk / Traditional,positive,0.297138,0.297138
6,Johnny Cash,Southwind,Country,negative,0.371507,-0.371507
7,Johnny Cash,Southwind,Folk / Traditional,negative,0.371507,-0.371507
8,Johnny Cash,Where We'll Never Grow Old,Country,positive,0.806687,0.806687
9,Johnny Cash,Where We'll Never Grow Old,Folk / Traditional,positive,0.806687,0.806687


#### **3.3. Agregación Sentimiento por género**

Consiste en agrupar las canciones por su estilo musical y promediar los resultados obtenidos por los modelos. Esta vista macro nos permite identificar tendencias generales y descubrir qué emociones o sentimientos predominan globalmente en cada género.

##### **3.3.1. Cardiff**

In [129]:

# ============================================================
# Métricas numéricas por género
# ============================================================

genre_sentiment_summary = (
    df_song_genre_models
    .groupby("genre")
    .agg(
        n_canciones=("song_norm", "count"),
        sentimiento_medio=("sentiment_signed_score", "mean"),
        sentimiento_mediano=("sentiment_signed_score", "median"),
        desviacion_tipica=("sentiment_signed_score", "std")
    )
    .reset_index()
)

# ============================================================
# Porcentajes de positive / negative / neutral por género
# ============================================================

sentiment_counts = pd.crosstab(
    df_song_genre_models["genre"],
    df_song_genre_models["cardiff_sentiment"],
    normalize="index"
) * 100

sentiment_counts = sentiment_counts.round(2)

# Asegurar que existen siempre las tres columnas, aunque alguna no aparezca
for sentiment in ["negative", "neutral", "positive"]:
    if sentiment not in sentiment_counts.columns:
        sentiment_counts[sentiment] = 0.0

# Reordenar columnas
sentiment_counts = sentiment_counts[["negative", "neutral", "positive"]]

# Sentimiento predominante por género
sentiment_counts["sentimiento_predominante"] = sentiment_counts.idxmax(axis=1)

# Renombrar columnas de porcentaje
sentiment_counts = (
    sentiment_counts
    .rename(columns={
        "negative": "pct_negative",
        "neutral": "pct_neutral",
        "positive": "pct_positive"
    })
    .reset_index()
)

# ============================================================
# Tabla final
# ============================================================

genre_sentiment_final = genre_sentiment_summary.merge(
    sentiment_counts,
    on="genre",
    how="left"
)

# Ordenar géneros alfabéticamente y crear índice de género 0, 1, 2...
genre_sentiment_final = (
    genre_sentiment_final
    .sort_values("genre", ascending=True)
    .reset_index(drop=True)
)

genre_sentiment_final.insert(
    0,
    "genre_id",
    range(len(genre_sentiment_final))
)

# Orden final de columnas
cols_order = [
    "genre",
    "sentimiento_predominante",
    "n_canciones",
    "sentimiento_medio",
    "sentimiento_mediano",
    "desviacion_tipica",
    "pct_negative",
    "pct_neutral",
    "pct_positive"
]

genre_sentiment_final = genre_sentiment_final[cols_order]

display(genre_sentiment_final)


,genre,sentimiento_predominante,n_canciones,sentimiento_medio,sentimiento_mediano,desviacion_tipica,pct_negative,pct_neutral,pct_positive
0,Blues,negative,8,-0.226155,-0.299953,0.467085,87.50,0.0,12.50
1,Classical / Art Music,positive,2,0.781275,0.781275,0.076284,0.00,0.0,100.00
2,Country,positive,524,0.101681,0.256776,0.581330,43.51,0.0,56.49
3,Electronic / Dance,positive,565,0.129701,0.305936,0.580209,40.88,0.0,59.12
4,Experimental / Other,positive,32,0.083875,0.211020,0.500733,37.50,0.0,62.50
5,Folk / Traditional,positive,257,0.061134,0.200445,0.574198,46.30,0.0,53.70
6,Hip Hop / Rap,negative,51,0.000780,-0.205346,0.536527,50.98,0.0,49.02
7,Jazz,positive,15,0.231811,0.392642,0.488999,40.00,0.0,60.00
8,Latin,positive,3,0.477696,0.439377,0.193216,0.00,0.0,100.00
9,Metal,negative,114,-0.265169,-0.448241,0.555693,71.93,0.0,28.07


#### **3.3.2. J-hartmann**

In [130]:

# ============================================================
# Porcentajes de emociones j-hartmann por género
# ============================================================

jhartmann_emotion_pct = pd.crosstab(
    df_song_genre_models["genre"],
    df_song_genre_models["jhartmann_emotion"],
    normalize="index"
) * 100

jhartmann_emotion_pct = jhartmann_emotion_pct.round(2)

# Asegurar que existen siempre las 7 columnas del modelo
jhartmann_emotions = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
]

for emotion in jhartmann_emotions:
    if emotion not in jhartmann_emotion_pct.columns:
        jhartmann_emotion_pct[emotion] = 0.0

# Reordenar columnas
jhartmann_emotion_pct = jhartmann_emotion_pct[jhartmann_emotions]

# Emoción predominante por género
jhartmann_emotion_pct["jhartmann_emotion_predominante"] = (
    jhartmann_emotion_pct.idxmax(axis=1)
)

# Renombrar columnas de porcentaje
jhartmann_emotion_pct = (
    jhartmann_emotion_pct
    .rename(columns={
        "anger": "pct_jhartmann_anger",
        "disgust": "pct_jhartmann_disgust",
        "fear": "pct_jhartmann_fear",
        "joy": "pct_jhartmann_joy",
        "neutral": "pct_jhartmann_neutral",
        "sadness": "pct_jhartmann_sadness",
        "surprise": "pct_jhartmann_surprise"
    })
    .reset_index()
)

# ============================================================
# Número de canciones por género
# ============================================================

jhartmann_counts = (
    df_song_genre_models
    .groupby("genre")
    .agg(n_canciones=("song_norm", "count"))
    .reset_index()
)

# ============================================================
# Tabla final
# ============================================================

genre_jhartmann_final = jhartmann_counts.merge(
    jhartmann_emotion_pct,
    on="genre",
    how="left"
)

# Ordenar géneros alfabéticamente y crear índice 0, 1, 2...
genre_jhartmann_final = (
    genre_jhartmann_final
    .sort_values("n_canciones", ascending=False)
    .reset_index(drop=True)
)

genre_jhartmann_final.insert(
    0,
    "genre_id",
    range(len(genre_jhartmann_final))
)

# Orden final de columnas
cols_order = [
    "genre",
    "jhartmann_emotion_predominante",
    "n_canciones",
    "pct_jhartmann_anger",
    "pct_jhartmann_disgust",
    "pct_jhartmann_fear",
    "pct_jhartmann_joy",
    "pct_jhartmann_neutral",
    "pct_jhartmann_sadness",
    "pct_jhartmann_surprise"
]

genre_jhartmann_final = genre_jhartmann_final[cols_order]

display(genre_jhartmann_final)



,genre,jhartmann_emotion_predominante,n_canciones,pct_jhartmann_anger,pct_jhartmann_disgust,pct_jhartmann_fear,pct_jhartmann_joy,pct_jhartmann_neutral,pct_jhartmann_sadness,pct_jhartmann_surprise
0,Rock,sadness,655,16.03,5.04,24.27,9.77,0.0,34.05,10.84
1,Electronic / Dance,sadness,565,14.16,2.83,22.30,17.35,0.0,31.68,11.68
2,Pop,sadness,542,11.62,2.58,24.91,17.16,0.0,33.58,10.15
3,Country,sadness,524,7.82,2.86,17.75,16.98,0.0,40.08,14.50
4,Folk / Traditional,sadness,257,12.84,2.33,18.29,14.79,0.0,39.69,12.06
5,Other,sadness,207,14.98,8.70,20.77,13.04,0.0,29.47,13.04
6,R&B / Soul / Funk,sadness,164,9.15,2.44,15.24,23.78,0.0,42.68,6.71
7,Punk / Emo / Hardcore,sadness,144,27.78,5.56,23.61,2.78,0.0,36.81,3.47
8,Metal,sadness,114,25.44,3.51,21.05,6.14,0.0,38.60,5.26
9,Hip Hop / Rap,anger,51,41.18,5.88,15.69,17.65,0.0,13.73,5.88


#### **3.3.3. GoEmotions**

In [131]:
# ============================================================
# GoEmotions: resumen por género + tabla compacta
# ============================================================

from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ============================================================
# 1. Preparar GoEmotions Top 3 en formato largo
# ============================================================

df_go_top3 = df_song_genre_models[
    ["artist", "song", "artist_norm", "song_norm", "genre", "goemotions_top3"]
].copy()

df_go_top3["goemotion"] = (
    df_go_top3["goemotions_top3"]
    .fillna("")
    .astype(str)
    .str.split(", ")
)

df_go_top3 = df_go_top3.explode("goemotion").reset_index(drop=True)

df_go_top3["goemotion"] = df_go_top3["goemotion"].astype(str).str.strip()

df_go_top3 = df_go_top3[
    df_go_top3["goemotion"].notna() &
    (df_go_top3["goemotion"] != "") &
    (df_go_top3["goemotion"].str.lower() != "nan")
].copy()

# ============================================================
# 2. Porcentajes de emociones GoEmotions por género
# ============================================================

goemotions = [
    "admiration",
    "amusement",
    "anger",
    "annoyance",
    "caring",
    "confusion",
    "curiosity",
    "desire",
    "disappointment",
    "disapproval",
    "disgust",
    "embarrassment",
    "excitement",
    "fear",
    "gratitude",
    "grief",
    "joy",
    "love",
    "nervousness",
    "optimism",
    "pride",
    "realization",
    "relief",
    "remorse",
    "sadness",
    "surprise",
    "neutral"
]

goemotion_counts_by_genre = pd.crosstab(
    df_go_top3["genre"],
    df_go_top3["goemotion"],
    normalize="index"
) * 100

goemotion_counts_by_genre = goemotion_counts_by_genre.round(2)

for emotion in goemotions:
    if emotion not in goemotion_counts_by_genre.columns:
        goemotion_counts_by_genre[emotion] = 0.0

goemotion_counts_by_genre = goemotion_counts_by_genre[goemotions]

goemotion_counts_by_genre["goemotions_emotion_predominante"] = (
    goemotion_counts_by_genre.idxmax(axis=1)
)

goemotion_counts_by_genre = (
    goemotion_counts_by_genre
    .rename(columns={emotion: f"pct_go_{emotion}" for emotion in goemotions})
    .reset_index()
)

# ============================================================
# 3. Tabla final completa
# ============================================================

goemotions_counts = (
    df_song_genre_models
    .groupby("genre")
    .agg(n_canciones=("song_norm", "count"))
    .reset_index()
)

genre_goemotions_final = goemotions_counts.merge(
    goemotion_counts_by_genre,
    on="genre",
    how="left"
)

genre_goemotions_final = (
    genre_goemotions_final
    .sort_values("n_canciones", ascending=False)
    .reset_index(drop=True)
)

cols_order = [
    "genre",
    "goemotions_emotion_predominante",
    "n_canciones"
] + [f"pct_go_{emotion}" for emotion in goemotions]

genre_goemotions_final = genre_goemotions_final[cols_order]

display(genre_goemotions_final)

# ============================================================
# 4. Tabla compacta: género, emoción predominante y top 3
# ============================================================

def get_top3_emotions_text(emotions):
    top3 = emotions.value_counts().head(3).index.tolist()
    return ", ".join(top3)


def get_top3_emotions_with_pct(emotions):
    counts = emotions.value_counts().head(3)
    total = len(emotions)

    return ", ".join([
        f"{emotion} ({round(count / total * 100, 2)}%)"
        for emotion, count in counts.items()
    ])


goemotions_top3_by_genre = (
    df_go_top3
    .groupby("genre")["goemotion"]
    .agg(
        top3_goemotions_mas_comunes=get_top3_emotions_text,
        top3_goemotions_mas_comunes_pct=get_top3_emotions_with_pct
    )
    .reset_index()
)

genre_goemotions_compact = (
    genre_goemotions_final[
        ["genre", "goemotions_emotion_predominante", "n_canciones"]
    ]
    .merge(
        goemotions_top3_by_genre,
        on="genre",
        how="left"
    )
    .sort_values("n_canciones", ascending=False)
    .reset_index(drop=True)
)

genre_goemotions_compact = genre_goemotions_compact[
    [
        "genre",
        "goemotions_emotion_predominante",
        "top3_goemotions_mas_comunes",
        "top3_goemotions_mas_comunes_pct"
    ]
]

display(genre_goemotions_compact)


,genre,goemotions_emotion_predominante,n_canciones,pct_go_admiration,pct_go_amusement,pct_go_anger,pct_go_annoyance,pct_go_caring,pct_go_confusion,pct_go_curiosity,...,pct_go_love,pct_go_nervousness,pct_go_optimism,pct_go_pride,pct_go_realization,pct_go_relief,pct_go_remorse,pct_go_sadness,pct_go_surprise,pct_go_neutral
0,Rock,disappointment,655,2.04,0.92,2.09,9.82,3.77,4.89,6.51,...,8.04,0.56,3.87,0.05,5.70,0.20,0.66,10.43,0.61,0.0
1,Electronic / Dance,love,565,3.19,1.12,2.71,8.61,4.72,3.72,5.49,...,12.80,0.53,2.77,0.00,4.19,0.00,0.53,8.14,0.65,0.0
2,Pop,love,542,3.01,1.29,1.60,7.07,4.12,3.32,4.61,...,14.94,0.80,2.64,0.00,3.32,0.00,0.55,10.21,0.62,0.0
3,Country,love,524,3.56,1.46,0.51,6.55,3.75,2.04,3.18,...,14.31,0.57,4.01,0.06,4.96,0.19,0.32,10.18,0.45,0.0
4,Folk / Traditional,love,257,3.37,0.78,1.17,7.65,3.76,3.50,5.45,...,12.19,0.65,3.37,0.00,5.06,0.26,0.52,10.12,0.78,0.0
5,Other,disappointment,207,3.38,0.81,1.93,8.21,4.67,5.31,8.05,...,8.86,0.00,3.86,0.00,3.70,0.00,0.32,9.02,0.97,0.0
6,R&B / Soul / Funk,love,164,2.03,1.42,1.02,5.28,8.33,1.63,3.46,...,16.46,0.20,5.89,0.00,2.85,0.00,0.61,10.57,0.00,0.0
7,Punk / Emo / Hardcore,sadness,144,1.85,0.69,3.70,12.50,4.40,5.32,7.41,...,5.32,1.62,3.01,0.23,2.78,0.00,1.39,13.66,0.23,0.0
8,Metal,sadness,114,1.75,0.29,4.97,11.99,3.22,5.26,7.31,...,6.73,1.75,2.92,0.00,5.56,0.00,0.29,13.16,0.58,0.0
9,Hip Hop / Rap,annoyance,51,1.31,3.92,12.42,18.95,3.27,3.27,3.27,...,7.19,0.00,1.96,0.00,4.58,0.00,0.00,3.92,0.65,0.0


,genre,goemotions_emotion_predominante,top3_goemotions_mas_comunes,top3_goemotions_mas_comunes_pct
0,Rock,disappointment,"approval, disappointment, sadness","approval (11.6%), disappointment (11.04%), sad..."
1,Electronic / Dance,love,"approval, love, annoyance","approval (14.34%), love (12.8%), annoyance (8...."
2,Pop,love,"love, approval, sadness","love (14.94%), approval (11.99%), sadness (10...."
3,Country,love,"love, approval, disappointment","love (14.31%), approval (12.21%), disappointme..."
4,Folk / Traditional,love,"love, approval, disappointment","love (12.19%), approval (11.28%), disappointme..."
5,Other,disappointment,"approval, disappointment, sadness","approval (13.37%), disappointment (9.18%), sad..."
6,R&B / Soul / Funk,love,"love, approval, sadness","love (16.46%), approval (12.4%), sadness (10.57%)"
7,Punk / Emo / Hardcore,sadness,"sadness, disappointment, annoyance","sadness (13.66%), disappointment (12.96%), ann..."
8,Metal,sadness,"sadness, annoyance, disappointment","sadness (13.16%), annoyance (11.99%), disappoi..."
9,Hip Hop / Rap,annoyance,"annoyance, approval, anger","annoyance (18.95%), approval (13.07%), anger (..."


---

### **4. Visualización resultados**



In [145]:
# Carpetas dentro de los outputs para organizar los archivos
# Carpeta principal
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Subcarpetas
CSV_DIR = OUTPUT_DIR / "csv"
HTML_DIR = OUTPUT_DIR / "html"

CSV_DIR.mkdir(exist_ok=True)
HTML_DIR.mkdir(exist_ok=True)


#### **4.1. Canciones por género**

Este gráfico de barras horizontales muestra el total de canciones únicas analizadas, clasificadas por género musical. Las barras se ordenan de menor a mayor cantidad, permitiendo comparar visualmente la representación de cada estilo en el dataset. Además, cada barra incluye el número exacto de pistas en su extremo derecho, facilitando la identificación rápida de los géneros musicales más y menos predominantes en la base de datos.

In [160]:
# ============================================================
# 1. Number of songs per genre - versión Plotly
# ============================================================

import pandas as pd
import plotly.express as px
import plotly.io as pio
from pathlib import Path


# ------------------------------------------------------------
# Renderer
# ------------------------------------------------------------

# Si usas VS Code:
pio.renderers.default = "vscode"

# Si estás en Jupyter Notebook clásico y no se muestra, prueba:
# pio.renderers.default = "notebook_connected"

# Si usas JupyterLab:
# pio.renderers.default = "jupyterlab"

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

df_genre_counts = df_song_genre_models.copy()

df_genre_counts = df_genre_counts.drop_duplicates(
    subset=["artist_norm", "song_norm", "genre"]
)

genre_counts = (
    df_genre_counts
    .groupby("genre")
    .agg(n_songs=("song_norm", "count"))
    .reset_index()
    .sort_values("n_songs", ascending=True)
)

# ------------------------------------------------------------
# Plotly chart
# ------------------------------------------------------------

fig = px.bar(
    genre_counts,
    x="n_songs",
    y="genre",
    orientation="h",
    text="n_songs",
    title="Number of Songs per Genre",
    labels={
        "n_songs": "Number of songs",
        "genre": "Genre"
    }
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    width=1000,
    height=700,
    title=dict(
        font=dict(size=20)
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray"
    ),
    yaxis=dict(
        title="Genre"
    ),
    margin=dict(l=160, r=80, t=80, b=60)
)

fig.show()

# ------------------------------------------------------------
# Save as HTML
# ------------------------------------------------------------

html_path = OUTPUT_DIR / "html/01_number_of_songs_per_genre_plotly.html"

fig.write_html(str(html_path))


# ------------------------------------------------------------
# Save as PNG with Matplotlib
# ------------------------------------------------------------

import matplotlib.pyplot as plt

fig_mpl, ax = plt.subplots(figsize=(12, 8))

ax.barh(
    genre_counts["genre"],
    genre_counts["n_songs"]
)

ax.set_title(
    "Number of Songs per Genre",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Number of songs")
ax.set_ylabel("Genre")

ax.grid(axis="x", linestyle="--", alpha=0.4)

for i, value in enumerate(genre_counts["n_songs"]):
    ax.text(
        value + 1,
        i,
        str(value),
        va="center",
        fontsize=10
    )

plt.tight_layout()

png_path = OUTPUT_DIR / "csv/01_number_of_songs_per_genre.png"

fig_mpl.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig_mpl)




#### **4.2. Sentimiento medio por género**

Este gráfico ilustra el sentimiento promedio por género musical, ordenándolos desde los más negativos a los más positivos. Cada barra indica la cantidad de canciones analizadas (n) y cruza un eje central en cero que separa claramente las puntuaciones pesimistas de las optimistas. Además, utiliza colores para destacar la emoción predominante, permitiendo identificar al instante qué estilos musicales tienden a ser más alegres o melancólicos.

In [161]:
# ============================================================
# 2. Mean sentiment by genre
# Mostrar con Plotly + guardar HTML + guardar PNG con Matplotlib
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
from matplotlib.patches import Patch
from pathlib import Path

# ------------------------------------------------------------
# Renderer de Plotly
# ------------------------------------------------------------

# Si usas VS Code:
pio.renderers.default = "vscode"

# Si no se muestra, prueba alternativamente:
# pio.renderers.default = "notebook_connected"
# pio.renderers.default = "jupyterlab"

# ------------------------------------------------------------
# Carpeta de salida
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# Preparar datos
# ------------------------------------------------------------

plot_data = genre_sentiment_final.copy()

plot_data["genre_label"] = (
    plot_data["genre"] + " (n=" + plot_data["n_canciones"].astype(str) + ")"
)

plot_data = plot_data.sort_values(
    "sentimiento_medio",
    ascending=True
).reset_index(drop=True)


# ------------------------------------------------------------
# Mostrar con Plotly
# ------------------------------------------------------------

color_map = {
    "negative": "#d73027",
    "neutral": "#fee08b",
    "positive": "#1a9850"
}

fig_plotly = px.bar(
    plot_data,
    x="sentimiento_medio",
    y="genre_label",
    orientation="h",
    color="sentimiento_predominante",
    text=plot_data["sentimiento_medio"].round(2),
    title="Sentimiento medio por género",
    labels={
        "sentimiento_medio": "Sentimiento medio",
        "genre_label": "Género",
        "sentimiento_predominante": "Sentimiento predominante"
    },
    color_discrete_map=color_map
)

fig_plotly.update_traces(
    textposition="outside"
)

# Quitar "neutral" de la leyenda, pero mantener sus barras
for trace in fig_plotly.data:
    if trace.name == "negative":
        trace.name = "Negativo"
    elif trace.name == "positive":
        trace.name = "Positivo"
    elif trace.name == "neutral":
        trace.name = "Neutral"
        trace.showlegend = False

fig_plotly.add_vline(
    x=0,
    line_width=1,
    line_dash="dash",
    line_color="black"
)

fig_plotly.update_layout(
    width=1100,
    height=800,
    title=dict(font=dict(size=20)),
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(title="Género"),
    legend=dict(
        title="Sentimiento predominante"
    ),
    margin=dict(l=200, r=80, t=80, b=60)
)

fig_plotly.show()

# ------------------------------------------------------------
# Guardar HTML con Plotly
# ------------------------------------------------------------

html_path = OUTPUT_DIR / "html/02_mean_sentiment_by_genre_plotly.html"

fig_plotly.write_html(str(html_path))

# ------------------------------------------------------------
# Guardar PNG con Matplotlib
# ------------------------------------------------------------

fig_mpl, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    data=plot_data,
    x="sentimiento_medio",
    y="genre_label",
    hue="sentimiento_predominante",
    dodge=False,
    palette="RdYlGn",
    ax=ax
)

ax.axvline(0, color="black", linewidth=1, linestyle="--")

ax.set_title(
    "Sentimiento medio por género",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Sentimiento medio")
ax.set_ylabel("Género")

# Quitar leyenda automática
if ax.get_legend() is not None:
    ax.get_legend().remove()

# Crear leyenda manual sin neutral
legend_elements = [
    Patch(facecolor=sns.color_palette("RdYlGn", 3)[0], label="Negativo"),
    Patch(facecolor=sns.color_palette("RdYlGn", 3)[-1], label="Positivo")
]

ax.legend(
    handles=legend_elements,
    title="Sentimiento predominante",
    loc="upper right"
)

ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()

png_path = OUTPUT_DIR / "csv/02_mean_sentiment_by_genre.png"

fig_mpl.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig_mpl)


**Sentimiento medio por género**

El gráfico muestra que la mayoría de géneros presentan un sentimiento medio positivo, aunque con intensidades diferentes. Classical / Art Music y Latin aparecen como los más positivos, pero tienen muy pocas canciones, por lo que sus resultados deben interpretarse con cautela. Entre los géneros con más representación, destacan R&B / Soul / Funk, Electronic / Dance, Pop y Country con tendencia positiva moderada. En cambio, Rock aparece ligeramente negativo, mientras que Metal y Punk / Emo / Hardcore muestran los valores más negativos, indicando una mayor carga emocional negativa en sus letras.

#### **4.3. Heatmap de sentimientos por género - Modelo J-hartmann**

Este mapa de calor visualiza la distribución porcentual de las siete emociones del modelo j-hartmann en cada género musical. Las celdas muestran el porcentaje exacto y utilizan una escala de colores cálidos donde los tonos más oscuros o intensos indican una mayor presencia emocional. Esta representación gráfica permite identificar de un solo vistazo qué emociones dominan en cada estilo y descubrir patrones clave en las letras.

In [ ]:
# ============================================================
# 4. j-hartmann heatmap by genre
# Mostrar con Plotly + guardar HTML + guardar PNG con Matplotlib
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
from pathlib import Path

# ------------------------------------------------------------
# Renderer de Plotly
# ------------------------------------------------------------

# Si usas VS Code:
pio.renderers.default = "vscode"

# Si no se muestra, prueba alternativamente:
# pio.renderers.default = "notebook_connected"
# pio.renderers.default = "jupyterlab"

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

plot_data = genre_jhartmann_final.copy()

MIN_SONGS = 1

plot_data = plot_data[plot_data["n_canciones"] >= MIN_SONGS].copy()

plot_data = plot_data.sort_values("n_canciones", ascending=False)

plot_data["genre_label"] = (
    plot_data["genre"] + " (n=" + plot_data["n_canciones"].astype(str) + ")"
)

jhartmann_emotions = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
]

jhartmann_pct_cols = [f"pct_jhartmann_{emotion}" for emotion in jhartmann_emotions]

for col in jhartmann_pct_cols:
    if col not in plot_data.columns:
        plot_data[col] = 0.0

heatmap_data = plot_data.set_index("genre_label")[jhartmann_pct_cols].copy()
heatmap_data.columns = jhartmann_emotions


# ============================================================
# Mostrar con Plotly
# ============================================================

fig_plotly = px.imshow(
    heatmap_data,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="YlOrRd",
    title="j-hartmann Emotion Distribution by Genre",
    labels={
        "x": "Emotion",
        "y": "Genre",
        "color": "Percentage (%)"
    }
)

fig_plotly.update_layout(
    width=1000,
    height=800,
    title=dict(font=dict(size=20)),
    margin=dict(l=220, r=80, t=80, b=80)
)

fig_plotly.update_xaxes(
    tickangle=45
)

fig_plotly.show()

# ------------------------------------------------------------
# Guardar HTML con Plotly
# ------------------------------------------------------------

html_path = OUTPUT_DIR / "html/04_jhartmann_heatmap_by_genre_plotly.html"

fig_plotly.write_html(str(html_path))


# ============================================================
# Guardar PNG con Matplotlib
# ============================================================

fig_mpl, ax = plt.subplots(figsize=(12, 8))

im = ax.imshow(
    heatmap_data.values,
    aspect="auto",
    cmap="YlOrRd"
)

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(
    heatmap_data.columns,
    rotation=45,
    ha="right"
)

ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

ax.set_title(
    "j-hartmann Emotion Distribution by Genre",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Emotion")
ax.set_ylabel("Genre")

cbar = fig_mpl.colorbar(im, ax=ax)
cbar.set_label("Percentage (%)")

max_value = heatmap_data.values.max()

for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        value = heatmap_data.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.1f}",
            ha="center",
            va="center",
            fontsize=8,
            color="black" if value < max_value * 0.65 else "white"
        )

plt.tight_layout()

png_path = OUTPUT_DIR / "csv/04_jhartmann_heatmap_by_genre.png"

fig_mpl.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig_mpl)


#### **4.4. Emociones predominantes por género**

Este panel compuesto por múltiples subgráficos ilustra las 5 emociones predominantes del modelo GoEmotions para nueve géneros musicales específicos. Mediante barras horizontales, detalla el porcentaje exacto de cada emoción dentro de estilos como el Rock, Pop o Metal. Esta vista facilita la comparación directa, revelando rápidamente el perfil emocional complejo y los matices sentimentales únicos que caracterizan a cada género musical.

In [164]:
# ============================================================
# 5. GoEmotions por género

# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from pathlib import Path

# ------------------------------------------------------------
# Renderer de Plotly
# ------------------------------------------------------------

# Si usas VS Code:
pio.renderers.default = "vscode"

# Si no se muestra, prueba alternativamente:
# pio.renderers.default = "notebook_connected"
# pio.renderers.default = "jupyterlab"

# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

target_genres = [
    "Rock",
    "Electronic / Dance",
    "Pop",
    "Country",
    "Folk / Traditional",
    "R&B / Soul / Funk",
    "Metal",
    "Hip Hop / Rap",
    "Punk / Emo / Hardcore"
]

TOP_N = 5

# ------------------------------------------------------------
# Detectar columnas de GoEmotions
# ------------------------------------------------------------

goemotion_pct_cols = [
    col for col in genre_goemotions_final.columns
    if col.startswith("pct_go_")
]

# Crear dataframe largo
plot_long = genre_goemotions_final[
    ["genre"] + goemotion_pct_cols
].copy()

plot_long = plot_long[plot_long["genre"].isin(target_genres)].copy()

plot_long = plot_long.melt(
    id_vars="genre",
    value_vars=goemotion_pct_cols,
    var_name="emotion",
    value_name="percentage"
)

plot_long["emotion"] = plot_long["emotion"].str.replace("pct_go_", "", regex=False)


# ============================================================
# Mostrar con Plotly
# ============================================================

fig_plotly = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=target_genres,
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

for i, genre_name in enumerate(target_genres):
    row = i // 3 + 1
    col = i % 3 + 1

    genre_data = plot_long[plot_long["genre"] == genre_name].copy()

    genre_data = genre_data.sort_values("percentage", ascending=False).head(TOP_N)
    genre_data = genre_data.sort_values("percentage", ascending=True)

    fig_plotly.add_trace(
        go.Bar(
            x=genre_data["percentage"],
            y=genre_data["emotion"],
            orientation="h",
            text=[f"{v:.1f}%" for v in genre_data["percentage"]],
            textposition="outside",
            showlegend=False
        ),
        row=row,
        col=col
    )

    fig_plotly.update_xaxes(
        title_text="% dentro del top 3",
        showgrid=True,
        gridcolor="lightgray",
        row=row,
        col=col
    )

    fig_plotly.update_yaxes(
        title_text="",
        row=row,
        col=col
    )

fig_plotly.update_layout(
    width=1400,
    height=1100,
    title=dict(
        text=f"Top {TOP_N} emociones GoEmotions por género",
        font=dict(size=20)
    ),
    margin=dict(l=80, r=80, t=100, b=60)
)

fig_plotly.show()

# ------------------------------------------------------------
# Guardar HTML con Plotly
# ------------------------------------------------------------

html_path = OUTPUT_DIR / "html/05_goemotions_top5_per_genre_plotly.html"

fig_plotly.write_html(str(html_path))


# ============================================================
# Guardar PNG con Matplotlib
# ============================================================

fig_mpl, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i, genre_name in enumerate(target_genres):
    ax = axes[i]

    genre_data = plot_long[plot_long["genre"] == genre_name].copy()

    genre_data = genre_data.sort_values("percentage", ascending=False).head(TOP_N)
    genre_data = genre_data.sort_values("percentage", ascending=True)

    ax.barh(
        genre_data["emotion"],
        genre_data["percentage"]
    )

    ax.set_title(genre_name, fontsize=13, fontweight="bold")
    ax.set_xlabel("% dentro del top 3 de GoEmotions")
    ax.set_ylabel("")
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    for y, value in enumerate(genre_data["percentage"]):
        ax.text(
            value + 0.5,
            y,
            f"{value:.1f}%",
            va="center",
            fontsize=9
        )

# Ocultar ejes sobrantes si los hubiera
for j in range(len(target_genres), len(axes)):
    axes[j].axis("off")

plt.suptitle(
    f"Top {TOP_N} emociones GoEmotions por género",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

png_path = OUTPUT_DIR / "csv/05_goemotions_top5_per_genre.png"

fig_mpl.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig_mpl)


**Emociones GoEmotions por género**

El gráfico permite observar perfiles emocionales más detallados por género. 

+ En `Rock` predominan emociones negativas o introspectivas como *disappointment, sadness y annoyance*, lo que encaja con el sentimiento medio ligeramente negativo visto antes. 

+ `Electronic / Dance`, `Pop`, `Country`, `Folk / Traditional` y `R&B / Soul / Funk` tienen *love* como una de las emociones principales, aunque también aparecen con fuerza sadness y disappointment, mostrando que incluso géneros positivos mezclan emociones afectivas con melancolía. 

+ `Metal` y `Punk / Emo / Hardcore` concentran emociones como *sadness, annoyance y disappointment*, reforzando su perfil más negativo. 

+ `Hip Hop / Rap` destaca especialmente por *annoyance y anger*, diferenciándose del resto por una carga más confrontativa. 

En general, GoEmotions muestra que los géneros no se distinguen solo por positividad o negatividad, sino por combinaciones emocionales específicas.

In [167]:
# ============================================================
# Evolución de positividad media por década
# Basado en Cardiff sentiment_signed_score
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
from pathlib import Path

# ------------------------------------------------------------
# Renderer de Plotly
# ------------------------------------------------------------

pio.renderers.default = "vscode"


# ------------------------------------------------------------
# Carpetas de salida
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_DIR = OUTPUT_DIR / "csv"
HTML_DIR = OUTPUT_DIR / "html"

CSV_DIR.mkdir(exist_ok=True)
HTML_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Preparar datos
# ------------------------------------------------------------

df_decade = df_model_comparison.copy()

# Una fila por canción
df_decade = df_decade.drop_duplicates(subset=["artist_norm", "song_norm"])

# Limpiar year
df_decade["year"] = pd.to_numeric(df_decade["year"], errors="coerce")

df_decade = df_decade[
    df_decade["year"].notna() &
    (df_decade["year"] >= 1900) &
    (df_decade["year"] <= 2030)
].copy()

df_decade["year"] = df_decade["year"].astype(int)

# Crear década
PERIOD = 10

df_decade["decade"] = (df_decade["year"] // PERIOD) * PERIOD
df_decade["decade_label"] = (
    df_decade["decade"].astype(str) + "-" + 
    (df_decade["decade"] + PERIOD - 1).astype(str)
)

# Crear score firmado si no existe
if "sentiment_signed_score" not in df_decade.columns:
    def sentiment_to_signed_score(row):
        label = row["cardiff_sentiment"]
        score = row["cardiff_sentiment_score"]

        if pd.isna(label) or pd.isna(score):
            return pd.NA

        label = str(label).lower()

        if label == "positive":
            return float(score)
        elif label == "negative":
            return -float(score)
        elif label == "neutral":
            return 0.0
        else:
            return pd.NA

    df_decade["sentiment_signed_score"] = df_decade.apply(
        sentiment_to_signed_score,
        axis=1
    )

df_decade = df_decade[df_decade["sentiment_signed_score"].notna()].copy()

print("Número de canciones válidas:", len(df_decade))


# ------------------------------------------------------------
# 2. Agregación por década
# ------------------------------------------------------------

decade_positivity = (
    df_decade
    .groupby(["decade", "decade_label"])
    .agg(
        n_songs=("song_norm", "count"),
        positividad_media=("sentiment_signed_score", "mean"),
        positividad_mediana=("sentiment_signed_score", "median"),
        desviacion_tipica=("sentiment_signed_score", "std")
    )
    .reset_index()
)

# Filtrar décadas con pocas canciones
MIN_SONGS_PER_DECADE = 0

decade_positivity = decade_positivity[
    decade_positivity["n_songs"] >= MIN_SONGS_PER_DECADE
].copy()

decade_positivity = decade_positivity.sort_values("decade").reset_index(drop=True)


# ------------------------------------------------------------
# 3. Mostrar con Plotly
# ------------------------------------------------------------

fig_plotly = px.line(
    decade_positivity,
    x="decade_label",
    y="positividad_media",
    markers=True,
    title="Evolution of Average Positivity by Decade",
    labels={
        "decade_label": "Decade",
        "positividad_media": "Average positivity score"
    },
    hover_data={
        "n_songs": True,
        "positividad_media": ":.3f",
        "positividad_mediana": ":.3f",
        "desviacion_tipica": ":.3f"
    }
)

fig_plotly.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

fig_plotly.update_layout(
    width=1000,
    height=650,
    title=dict(font=dict(size=20)),
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
    margin=dict(l=80, r=60, t=80, b=60)
)

fig_plotly.show()

# ------------------------------------------------------------
# 4. Guardar HTML
# ------------------------------------------------------------

html_path = HTML_DIR / "decade_average_positivity.html"
fig_plotly.write_html(str(html_path))


# ------------------------------------------------------------
# 5. Guardar PNG con Matplotlib
# ------------------------------------------------------------

fig_mpl, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    decade_positivity["decade_label"],
    decade_positivity["positividad_media"],
    marker="o",
    label="Average positivity"
)

ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

ax.set_title(
    "Evolution of Average Positivity by Decade",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Decade")
ax.set_ylabel("Average positivity score")

ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()

plt.tight_layout()

png_path = OUTPUT_DIR / "decade_average_positivity.png"

fig_mpl.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig_mpl)


Número de canciones válidas: 415


**Evolución de positividad media por década**

La evolución temporal indica que la positividad media de las canciones se mantiene siempre por encima de cero, por lo que el dataset presenta una tendencia general positiva en todas las décadas. 

Sin embargo, no hay una progresión lineal: la positividad es alta en los años 50, baja en los 60, vuelve a subir en los 70, cae en los 80 y alcanza su punto más bajo en la década de 2010. En los años 2020 se observa una recuperación clara. La interpretación debe considerar el número de canciones por década, ya que décadas con pocos ejemplos pueden producir valores más inestables.